Data needs to be scaled. (The data in this work is already scaled)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from yellowbrick.cluster import KElbowVisualizer
from sklearn.cluster import KMeans, AgglomerativeClustering
import pickle


from sklearn.cluster import KMeans, AffinityPropagation, MeanShift, estimate_bandwidth, OPTICS, DBSCAN
from sklearn.mixture import GaussianMixture
from itertools import product

In [20]:
df_processed = pd.read_csv("/content/drive/MyDrive/YL_Dersler/Python/Ödev/Codes/data_processed.csv")
df_processed.head()

,Education,Marital_Status,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,Response,Age,Customer_Tenure,Recency_Group,Tenure_Group,Total_Kids,Total_Spending,Total_Accepted_Cmp,Total_Num_Purchases,Income_Cat
0,-0.351516,0.259792,0.279789,-0.829097,-0.932051,0.312088,0.984044,1.554193,1.704658,2.464511,...,2.361161,0.990090,1.536426,-1.340362,0.445046,-1.275744,1.677075,-0.442817,1.322276,-1.330855
1,-0.351516,0.259792,-0.265715,1.031625,0.898255,-0.377754,-0.870634,-0.634978,-0.722194,-0.646539,...,-0.423521,1.241874,-1.184795,-0.449385,1.337741,1.396501,-0.961089,-0.442817,-1.157634,0.414223
2,-0.351516,1.189620,0.903044,-0.829097,-0.932051,-0.791659,0.362846,0.572841,-0.178400,1.348193,...,-0.423521,0.318666,-0.200208,-0.449385,-0.447649,-1.275744,0.281669,-0.442817,0.800190,1.286761
3,-0.351516,1.189620,-1.176801,1.031625,-0.932051,-0.791659,-0.870634,-0.559489,-0.659276,-0.500137,...,-0.423521,-1.275967,-1.056155,-0.449385,1.337741,0.060379,-0.917949,-0.442817,-0.896591,-0.458316
4,1.421572,-0.670036,0.286958,1.031625,-0.932051,1.553804,-0.389131,0.421863,-0.218847,0.158674,...,-0.423521,-1.024183,-0.947307,0.441592,1.337741,0.060379,-0.305696,-0.442817,0.539147,-1.330855


In [26]:
df_raw = pd.read_csv("/content/drive/MyDrive/YL_Dersler/Python/Ödev/Codes/marketing_campaign.csv", sep="\t")
df_raw.head()


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [27]:
numeric_cols = df_raw.select_dtypes(include='number').columns  # numerical columns
cat_col_names = df_raw.select_dtypes(include=['object', 'category']).columns


df_raw[numeric_cols] = df_raw[numeric_cols].fillna(df_raw[numeric_cols].median())

from sklearn.preprocessing import LabelEncoder

le2 = LabelEncoder()
for col in cat_col_names:
    df_raw[col] = le2.fit_transform(df_raw[col].astype(str))



from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler2 = StandardScaler()

# Fit the scaler to the data and transform it
df_raw_scaled = scaler2.fit_transform(df_raw)

### **Raw Data PCA**

In [32]:
from sklearn.decomposition import PCA


# Fit PCA first (X is your scaled data / features)
pca = PCA().fit(df_raw_scaled)

# Cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Targets
thresholds = [0.70, 0.80, 0.90, 0.95]

for thresh in thresholds:
    # number of components needed to reach threshold
    n_components = np.argmax(cumulative_variance >= thresh) + 1
    print(f"{int(thresh*100)}% = {n_components} components")

70% = 11 components
80% = 15 components
90% = 20 components
95% = 23 components


In [36]:
pca = PCA(n_components=23)
pca.fit(df_raw_scaled)

# Transform
pca_array = pca.transform(df_raw_scaled)

# Convert to DataFrame for clarity
pca_df_raw = pd.DataFrame(
    pca_array,
    columns=[f'PC{i+1}' for i in range(23)]
)

pca_df_raw

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23
0,3.770262,-0.329079,-0.421171,1.940913,-0.648734,0.629717,0.291842,1.067583,-0.572324,-0.094953,...,-2.629067,-1.601582,-0.693049,-0.788413,-0.698398,-0.044350,0.329824,0.564537,-1.006443,0.774818
1,-2.361511,0.193690,-0.215894,-0.931830,-0.463448,-0.057487,0.519373,0.635957,-0.408539,-1.642351,...,0.094507,-0.111341,0.186051,0.299149,0.214814,-0.252237,0.348738,-0.010370,-0.203659,-0.055259
2,1.621760,-0.155850,-1.090958,-0.209743,-0.021228,-0.019149,-0.297319,0.546316,1.443045,-0.380117,...,0.058288,0.591337,-0.195230,-0.044249,-0.417930,-0.640583,0.610912,-1.094726,0.267377,0.636193
3,-2.502090,-1.461546,0.197433,0.058030,0.104290,0.898884,-0.598272,1.145021,0.126173,0.093039,...,0.396620,0.397168,0.115952,-0.081418,0.035598,-0.211360,-0.128589,-0.040156,0.273163,-0.077151
4,-0.436875,0.029665,-0.452774,0.344999,0.727662,-1.022217,-0.507735,-0.283197,-1.426594,1.959907,...,-0.550656,0.366449,-0.167545,0.104972,0.312618,0.016367,0.181163,-0.162648,0.243219,-0.190088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,2.337431,0.999397,-1.518176,1.593405,0.176391,1.057856,-0.069055,-0.468516,0.557867,0.116371,...,-0.736826,0.194401,1.814245,0.555021,1.061737,0.636637,0.904200,1.636951,-0.435794,-0.590275
2236,-0.920747,2.922286,1.329642,0.324834,-0.077844,0.189468,0.370022,1.345990,-1.030418,0.717172,...,-0.249946,3.098744,0.462744,-0.935336,0.138303,-1.174335,1.019724,-0.674119,0.218690,-0.810649
2237,1.444116,0.113104,0.949020,-1.263546,2.612065,-0.385687,-1.069823,-1.796209,-0.371721,0.927972,...,0.620516,-1.227590,-2.042421,-0.651870,1.067777,-0.301279,-0.608693,-1.020358,0.789590,-0.146080
2238,1.766279,1.251830,-1.221960,-0.824666,-1.306809,0.520251,-0.358472,0.205711,1.795424,0.079689,...,0.521092,0.350241,0.134258,-0.177486,0.059647,-0.235385,-0.061130,-0.126123,0.317225,0.420958


### **Processed Data PCA**

In [37]:
from sklearn.decomposition import PCA


# Fit PCA first (X is your scaled data / features)
pca = PCA().fit(df_processed)

# Cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Targets
thresholds = [0.70, 0.80, 0.90, 0.95]

for thresh in thresholds:
    # number of components needed to reach threshold
    n_components = np.argmax(cumulative_variance >= thresh) + 1
    print(f"{int(thresh*100)}% = {n_components} components")

70% = 10 components
80% = 15 components
90% = 20 components
95% = 24 components


In [38]:
pca = PCA(n_components=20)
pca.fit(df_processed)

# Transform
pca_array = pca.transform(df_processed)

# Convert to DataFrame for clarity
pca_df_processed = pd.DataFrame(
    pca_array,
    columns=[f'PC{i+1}' for i in range(20)]
)

pca_df_processed

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20
0,4.224112,1.234213,-1.051216,2.335170,0.469611,-0.166696,-0.173486,1.288725,0.193267,0.480709,0.738108,-1.013048,3.229555,0.588564,-0.412943,-0.597737,-0.327493,0.512445,0.372179,0.460032
1,-3.055825,-0.475059,-0.072767,-2.100163,-0.300863,1.231707,0.464639,-0.397146,0.685282,0.480098,-0.362358,-0.204703,0.821787,-0.138061,-0.144846,0.093396,-0.449085,-0.443194,-0.123732,0.282330
2,2.071546,-0.042858,-1.656087,0.035352,-0.248836,-1.009198,-0.213010,0.000725,0.357933,0.814715,0.346242,-0.068959,-0.587848,-0.005880,0.999892,-0.082929,-0.268969,-1.666486,-0.597592,0.358140
3,-2.955772,-1.584185,-0.428560,-0.102019,-0.179048,0.311618,-1.107965,-0.263199,-0.385803,1.200112,1.362693,0.022085,-0.038622,0.588944,-0.163888,0.214597,-0.250631,-0.540016,-0.294632,0.182694
4,-0.628678,0.570869,-0.478209,-0.530926,0.283531,0.977964,-0.951209,0.147611,-2.082508,-0.344068,1.753256,0.543543,0.278542,0.408815,0.474379,-0.234309,0.583753,1.532234,0.864347,0.571548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,-0.673821,-0.431970,-0.403603,0.152060,-1.143657,-0.205909,1.854902,-1.633738,0.089605,0.863584,0.327557,1.113088,-1.616488,-0.648256,0.117536,-0.086695,-0.795513,-0.562276,0.370427,-0.192100
2054,2.267630,2.302745,-0.929678,1.216006,0.320460,0.192293,-0.894214,0.616605,0.502282,-0.198347,-1.754664,-0.083302,-0.106371,0.461331,-0.023626,2.589747,1.491377,-0.434374,0.424972,-0.344754
2055,1.969291,-0.723160,0.878611,-1.228699,1.025844,-1.309093,-3.298887,-0.553931,-1.716538,-1.435131,0.735188,-0.652977,-0.377751,-0.930527,-1.076360,-1.503391,1.062390,0.427171,1.467321,-0.040383
2056,1.961441,0.873678,-1.039218,-1.977342,-2.453885,0.240679,-0.003259,-0.202438,0.518170,1.057083,0.438993,0.256810,-0.169899,-0.591226,0.748627,0.089507,-0.253497,-0.663263,-0.208761,-0.276447


## **Hyperparameter Tuning**

In [8]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.base import clone
from itertools import product
import pandas as pd
import numpy as np

def clustering_grid_search(df, model_grid, filter_invalid=True, sort_by="Silhouette"):
    """
    Perform a grid search over multiple clustering models and hyperparameters.

    Parameters:
    - df : pd.DataFrame or np.array
        Scaled data. IMPORTANT: Distance-based models (KMeans, Agglomerative, GMM) require scaled data.
    - model_grid : dict
        Format: {
            "ModelName": (ModelClass, param_grid_dict)
        }
        Example:
        {
            "KMeans": (KMeans, {"n_clusters": [2,3,4]}),
            "Agglomerative": (AgglomerativeClustering, {"n_clusters":[2,3], "linkage":["ward","complete","average"]}),
            "DBSCAN": (DBSCAN, {"eps":[0.5,1], "min_samples":[3,5]}),
            "GMM": (GaussianMixture, {"n_components":[2,3], "covariance_type":["full","diag"]})
        }
    - filter_invalid : bool, default True
        If True, removes runs where Num_Clusters <= 1
    - sort_by : str, default "Silhouette"
        Column name to sort results by descending order (higher is better)

    Returns:
    - pd.DataFrame with all combinations and metrics
    """
    results = []

    for name, (ModelClass, param_grid) in model_grid.items():
        keys = list(param_grid.keys())
        values = list(param_grid.values())

        for combination in product(*values):
            params = dict(zip(keys, combination))

            # Add random_state if applicable
            if name in ["KMeans", "GMM"] and "random_state" not in params:
                params["random_state"] = 42

            model = ModelClass(**params)
            try:
                # Fit model and get labels
                if name == "GMM":
                    labels = model.fit(df).predict(df)
                else:
                    labels = model.fit_predict(df)

                # Handle DBSCAN noise
                unique_labels = set(labels)
                if -1 in unique_labels:
                    num_clusters = len(unique_labels) - 1
                    outlier_ratio = np.sum(labels == -1) / len(labels)
                else:
                    num_clusters = len(unique_labels)
                    outlier_ratio = 0.0

                # Compute metrics
                if num_clusters <= 1:
                    silhouette = np.nan
                    db_index = np.nan
                    ch_score = np.nan
                else:
                    silhouette = silhouette_score(df, labels)
                    db_index = davies_bouldin_score(df, labels)
                    ch_score = calinski_harabasz_score(df, labels)

                results.append({
                    "Model": name,
                    **params,
                    "Num_Clusters": num_clusters,
                    "Outlier_Ratio": outlier_ratio,
                    "Silhouette": silhouette,
                    "Davies-Bouldin": db_index,
                    "Calinski-Harabasz": ch_score
                })

            except Exception as e:
                print(f"{name} with params {params} failed: {e}")

    results_df = pd.DataFrame(results)

    # Optional filtering
    if filter_invalid:
        results_df = results_df[results_df["Num_Clusters"] > 1]

    # Sort by chosen metric
    if sort_by in results_df.columns:
        results_df = results_df.sort_values(by=sort_by, ascending=False).reset_index(drop=True)

    return results_df

In [16]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture

# Define parameter grids for each model
model_grid = {
    "KMeans": (KMeans, {"n_clusters": list(range(2,8))}),
    "Agglomerative": (AgglomerativeClustering, {"n_clusters": list(range(2,6)),
                                                "linkage": ["ward","complete","average"]}),
    "DBSCAN": (DBSCAN, {"eps":[1.0, 6.0, 1.0], "min_samples":[3,5]}),
    "GMM": (GaussianMixture, {"n_components": list(range(2,6)),
                              "covariance_type":["full","diag"]})
}

# Run grid search
results_df_processed = clustering_grid_search(df_processed, model_grid)
results_df_processed.head(50).sort_values(by=['Model']).style.background_gradient(cmap="Blues",
                                                        subset=results_df_processed.columns[5:8])

,Model,n_clusters,random_state,Num_Clusters,Outlier_Ratio,Silhouette,Davies-Bouldin,Calinski-Harabasz,linkage,eps,min_samples,n_components,covariance_type
0,Agglomerative,2.000000,nan,2,0.000000,0.447364,1.049757,99.159875,average,nan,nan,nan,nan
1,Agglomerative,2.000000,nan,2,0.000000,0.447364,1.049757,99.159875,complete,nan,nan,nan,nan
2,Agglomerative,3.000000,nan,3,0.000000,0.402059,1.027328,85.423810,complete,nan,nan,nan,nan
3,Agglomerative,3.000000,nan,3,0.000000,0.402059,1.027328,85.423810,average,nan,nan,nan,nan
4,Agglomerative,4.000000,nan,4,0.000000,0.373059,1.026411,61.554852,average,nan,nan,nan,nan
5,Agglomerative,5.000000,nan,5,0.000000,0.368389,0.907208,46.993418,average,nan,nan,nan,nan
22,Agglomerative,4.000000,nan,4,0.000000,0.107941,2.283153,265.599949,ward,nan,nan,nan,nan
7,Agglomerative,4.000000,nan,4,0.000000,0.311556,1.314419,126.166698,complete,nan,nan,nan,nan
8,Agglomerative,5.000000,nan,5,0.000000,0.308970,1.224177,99.127149,complete,nan,nan,nan,nan
21,Agglomerative,3.000000,nan,3,0.000000,0.110352,2.351621,320.864331,ward,nan,nan,nan,nan


In [31]:
# Run grid search
results_df_raw = clustering_grid_search(df_raw_scaled, model_grid)
results_df_raw.head(50).sort_values(by=['Model']).style.background_gradient(cmap="Blues", subset=results_df_raw.columns[5:8])

,Model,n_clusters,random_state,Num_Clusters,Outlier_Ratio,Silhouette,Davies-Bouldin,Calinski-Harabasz,linkage,eps,min_samples,n_components,covariance_type
0,Agglomerative,2.000000,nan,2,0.000000,0.725943,0.196191,23.037071,average,nan,nan,nan,nan
1,Agglomerative,2.000000,nan,2,0.000000,0.725943,0.196191,23.037071,complete,nan,nan,nan,nan
2,Agglomerative,3.000000,nan,3,0.000000,0.515747,0.591238,23.188719,complete,nan,nan,nan,nan
3,Agglomerative,3.000000,nan,3,0.000000,0.515747,0.591238,23.188719,average,nan,nan,nan,nan
4,Agglomerative,4.000000,nan,4,0.000000,0.453770,0.684544,45.785085,average,nan,nan,nan,nan
5,Agglomerative,5.000000,nan,5,0.000000,0.447487,0.794533,65.787397,average,nan,nan,nan,nan
22,Agglomerative,3.000000,nan,3,0.000000,0.093204,2.685025,281.301288,ward,nan,nan,nan,nan
7,Agglomerative,5.000000,nan,5,0.000000,0.407015,1.024831,59.573178,complete,nan,nan,nan,nan
8,Agglomerative,4.000000,nan,4,0.000000,0.405471,1.022962,38.335494,complete,nan,nan,nan,nan
21,Agglomerative,4.000000,nan,4,0.000000,0.103973,2.253901,240.914439,ward,nan,nan,nan,nan


In [42]:
# Run grid search
results_pca_df_processed = clustering_grid_search(pca_df_processed, model_grid)
results_pca_df_processed.head(50).sort_values(by=['Model']).style.background_gradient(cmap="Blues", subset=results_pca_df_processed.columns[5:8])

,Model,n_clusters,random_state,Num_Clusters,Outlier_Ratio,Silhouette,Davies-Bouldin,Calinski-Harabasz,linkage,eps,min_samples,n_components,covariance_type
0,Agglomerative,2.000000,nan,2,0.000000,0.463671,0.982245,109.937589,average,nan,nan,nan,nan
1,Agglomerative,2.000000,nan,2,0.000000,0.425503,0.957639,72.307923,complete,nan,nan,nan,nan
2,Agglomerative,3.000000,nan,3,0.000000,0.423671,0.967506,95.064901,complete,nan,nan,nan,nan
3,Agglomerative,3.000000,nan,3,0.000000,0.423671,0.967506,95.064901,average,nan,nan,nan,nan
19,Agglomerative,5.000000,nan,5,0.000000,0.159503,2.124584,266.191891,ward,nan,nan,nan,nan
17,Agglomerative,3.000000,nan,3,0.000000,0.180969,2.008351,376.334218,ward,nan,nan,nan,nan
6,Agglomerative,4.000000,nan,4,0.000000,0.326659,0.996503,67.191151,average,nan,nan,nan,nan
7,Agglomerative,5.000000,nan,5,0.000000,0.325517,0.883369,51.253199,average,nan,nan,nan,nan
8,Agglomerative,4.000000,nan,4,0.000000,0.263080,1.323228,245.665194,complete,nan,nan,nan,nan
9,Agglomerative,5.000000,nan,5,0.000000,0.261403,1.290715,187.622058,complete,nan,nan,nan,nan


In [43]:
# Run grid search
results_pca_df_raw = clustering_grid_search(pca_df_raw, model_grid)
results_pca_df_raw.head(50).sort_values(by=['Model']).style.background_gradient(cmap="Blues", subset=results_pca_df_raw.columns[5:8])

,Model,n_clusters,random_state,Num_Clusters,Outlier_Ratio,Silhouette,Davies-Bouldin,Calinski-Harabasz,linkage,eps,min_samples,n_components,covariance_type
0,Agglomerative,2.000000,nan,2,0.000000,0.680909,0.230315,16.719419,average,nan,nan,nan,nan
15,Agglomerative,3.000000,nan,3,0.000000,0.170015,2.281743,310.384353,ward,nan,nan,nan,nan
13,Agglomerative,5.000000,nan,5,0.000000,0.186685,1.689413,231.489246,ward,nan,nan,nan,nan
12,Agglomerative,2.000000,nan,2,0.000000,0.187040,2.132139,436.196874,ward,nan,nan,nan,nan
7,Agglomerative,5.000000,nan,5,0.000000,0.453440,0.775544,66.305484,average,nan,nan,nan,nan
6,Agglomerative,5.000000,nan,5,0.000000,0.453440,0.775544,66.305484,complete,nan,nan,nan,nan
14,Agglomerative,4.000000,nan,4,0.000000,0.178958,1.917360,256.470360,ward,nan,nan,nan,nan
4,Agglomerative,4.000000,nan,4,0.000000,0.459728,0.672687,44.814123,complete,nan,nan,nan,nan
5,Agglomerative,4.000000,nan,4,0.000000,0.459728,0.672687,44.814123,average,nan,nan,nan,nan
1,Agglomerative,2.000000,nan,2,0.000000,0.680909,0.230315,16.719419,complete,nan,nan,nan,nan


In [47]:
import pandas as pd

# List of datasets
datasets = {
    "Raw Data": results_df_raw,
    "Processed Data": results_df_processed,
    "Raw PCA": results_pca_df_raw,
    "Processed PCA": results_pca_df_processed
}

summary_list = []

for name, df in datasets.items():
    # Pick the best silhouette per clustering method
    best_df = df.loc[df.groupby('Model')['Silhouette'].idxmax(),
                     ['Model','Num_Clusters','Silhouette','Calinski-Harabasz','Davies-Bouldin']]

    # Add Dataset column
    best_df['Dataset'] = name

    summary_list.append(best_df)

# Concatenate all datasets
final_summary = pd.concat(summary_list, ignore_index=True)

# Reorder columns for readability
final_summary = final_summary[['Dataset','Model','Num_Clusters','Silhouette',
                               'Calinski-Harabasz','Davies-Bouldin']]

# Optional: Round numeric values for presentation
final_summary[['Silhouette','Calinski-Harabasz','Davies-Bouldin']] = final_summary[['Silhouette','Calinski-Harabasz','Davies-Bouldin']].round(4)

# Display final table
final_summary

,Dataset,Model,Num_Clusters,Silhouette,Calinski-Harabasz,Davies-Bouldin
0,Raw Data,Agglomerative,2,0.4637,109.9376,0.9822
1,Raw Data,DBSCAN,4,0.4074,47.8092,1.6034
2,Raw Data,GMM,2,0.2250,539.9599,1.8619
3,Raw Data,KMeans,2,0.2456,620.2171,1.7333
4,Processed Data,Agglomerative,2,0.4474,99.1599,1.0498
5,Processed Data,DBSCAN,4,0.3555,39.0988,1.9162
6,Processed Data,GMM,2,0.3023,302.2564,1.8179
7,Processed Data,KMeans,2,0.2318,546.7887,1.8572
8,Raw PCA,Agglomerative,2,0.6809,16.7194,0.2303
9,Raw PCA,DBSCAN,3,0.4441,67.6256,2.3130


## **Compare Best Models**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import pandas as pd


models = {
    "KMeans": KMeans(n_clusters=3, random_state=42),
    "GMM": GaussianMixture(n_components=3, random_state=42),
    "Agglomerative": AgglomerativeClustering(n_clusters=3),
    "DBSCAN": DBSCAN(eps=1, min_samples=3)
}



def compare_clustering_models(df, models, scale=True):
    # Optional scaling
    if scale:
        scaler = StandardScaler()
        X = scaler.fit_transform(df)
    else:
        X = df.values if isinstance(df, pd.DataFrame) else df

    results = []

    for name, model in models.items():
        try:
            labels = model.fit_predict(X)

            # Detect clusters / noise
            unique_labels = set(labels)
            if -1 in unique_labels:
                num_clusters = len(unique_labels) - 1
                noise_ratio = np.sum(labels == -1) / len(labels)
            else:
                num_clusters = len(unique_labels)
                noise_ratio = 0.0

            # Check if valid clustering
            if num_clusters <= 1:
                silhouette = np.nan
                db_index = np.nan
                ch_score = np.nan
            else:
                silhouette = silhouette_score(X, labels)
                db_index = davies_bouldin_score(X, labels)
                ch_score = calinski_harabasz_score(X, labels)

            results.append({
                'Model': name,
                'Silhouette': silhouette,
                'Davies-Bouldin': db_index,
                'Calinski-Harabasz': ch_score,
                'Num_Clusters': num_clusters,
                'Noise_Ratio': noise_ratio
            })

        except Exception as e:
            print(f"{name} failed: {e}")

    return pd.DataFrame(results)


compare_clustering_models(df_processed, models, None)

,Model,Silhouette,Davies-Bouldin,Calinski-Harabasz,Num_Clusters,Noise_Ratio
0,KMeans,0.196030,2.067841,391.325435,3,0.000000
1,GMM,0.192523,2.333046,334.759728,3,0.000000
2,Agglomerative,0.110352,2.351621,320.864331,3,0.000000
3,DBSCAN,-0.211725,1.418040,1.859612,3,0.995141
